# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khaled-dragon/ML-intern/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

This is a yes/no classification with an observed label (is_declining), so
per the training-honest-models guide, I'll start with Logistic Regression
(readable baseline) and compare it to Random Forest (stronger, still
interpretable via feature importance). I'm avoiding Gradient Boosting for
now, since the skill notes say to add complexity only when the comparison
earns it, and I want to see if the simpler models already beat my Week-4
rule baseline before reaching for something heavier.

In [1]:
%pip -q install duckdb huggingface_hub
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your HF READ token (hf_...): ··········


### "##############################################################################"

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [4]:
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

data = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month='2026-03'),
    prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               AVG(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY THEN gsc_avg_position END) AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month = '2026-03'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    ),
    outcome AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS imp_next30
        FROM {TABLES['fact_daily']} WHERE month = '2026-04'
        GROUP BY 1
    )
    SELECT p.*, o.imp_next30
    FROM prior p JOIN outcome o USING (content_hash_id)
""").df()

data['is_declining'] = (data['imp_next30'] < 0.8 * data['imp_prev30']).astype(int)
print(f"{len(data):,} rows, {data['client_hash_id'].nunique()} unique clients")
print(data['is_declining'].value_counts())

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.25, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data['client_hash_id']))
train, test = data.iloc[train_idx], data.iloc[test_idx]
print(f"Train: {len(train):,} rows, {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test):,} rows, {test['client_hash_id'].nunique()} clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

100,849 rows, 44 unique clients
is_declining
0    50626
1    50223
Name: count, dtype: int64
Train: 92,560 rows, 33 clients
Test:  8,289 rows, 11 clients


I used GroupShuffleSplit on client_hash_id instead of a random row split.
Without this, the same client's pages could appear in both train and test,
letting the model "cheat" by learning client-specific patterns rather than
generalizable ones. This matches the same client-holdout idea used in the
reference pipeline (scripts/03_train_model.py) from Week 1.

### "##############################################################################"

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Attempt 1 — baseline features only (imp_prev30, pos_prev30)

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

feature_cols = ['imp_prev30', 'pos_prev30']
X_train, y_train = train[feature_cols], train['is_declining']
X_test, y_test = test[feature_cols], test['is_declining']

def precision_at_k(scores, y_true, k):
    order = scores.argsort()[::-1][:k]
    return y_true.iloc[order].mean()

# My Week-4 rule baseline, recomputed here for direct comparison
def rule_score(imp, pos):
    volume_score = 1 - abs(imp - 500) / 5000
    position_score = max(0, (50 - pos) / 50) if pos <= 50 else 0
    return 0.5 * volume_score + 0.5 * position_score

baseline_scores = test.apply(lambda r: rule_score(r['imp_prev30'], r['pos_prev30']), axis=1)

log_reg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
log_scores = log_reg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

import pandas as pd
results = pd.DataFrame({
    'method': ['Week-4 rule baseline', 'Logistic Regression', 'Random Forest'],
    'precision@20': [
        precision_at_k(baseline_scores.values, y_test, 20),
        precision_at_k(log_scores, y_test, 20),
        precision_at_k(rf_scores, y_test, 20),
    ],
    'precision@50': [
        precision_at_k(baseline_scores.values, y_test, 50),
        precision_at_k(log_scores, y_test, 50),
        precision_at_k(rf_scores, y_test, 50),
    ],
})
print(f"base rate (share declining in test): {y_test.mean():.3f}")
print(results)

base rate (share declining in test): 0.471
                 method  precision@20  precision@50
0  Week-4 rule baseline          0.50          0.42
1   Logistic Regression          0.35          0.40
2         Random Forest          0.45          0.38


With only imp_prev30 and pos_prev30 (the same two inputs the rule uses),
the Week-4 rule baseline beat both models at precision@20 (0.50 vs 0.35/0.45)
and precision@50 (0.42 vs 0.40/0.38). This is an honest finding, not a bug:
with the exact same two features the rule already exploits well, a model
has nothing extra to learn from. I'm now adding query-mix features
(visible_queries, rare_share, top_query_share) that the rule never had
access to, to see if genuinely new information changes the result.

### Attempt 2 — adding query-mix features (leakage found, then re-checked)

In [6]:
TABLES['fact_query_90d'] = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           ANY_VALUE(rare_impressions_share)       AS rare_share,
           MAX(impressions_90d)                    AS top_query_impressions,
           SUM(impressions_90d)                    AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()
qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

data2 = data.merge(qsignals, on='content_hash_id', how='left').dropna(
    subset=['visible_queries', 'rare_share', 'top_query_share'])

train2 = data2[data2['client_hash_id'].isin(train['client_hash_id'])]
test2  = data2[data2['client_hash_id'].isin(test['client_hash_id'])]

feature_cols2 = ['imp_prev30', 'pos_prev30', 'visible_queries', 'rare_share', 'top_query_share']
X_train2, y_train2 = train2[feature_cols2], train2['is_declining']
X_test2,  y_test2  = test2[feature_cols2],  test2['is_declining']

log_reg2 = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
rf2 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X_train2, y_train2)

baseline_scores2 = test2.apply(lambda r: rule_score(r['imp_prev30'], r['pos_prev30']), axis=1)
log_scores2 = log_reg2.predict_proba(X_test2)[:, 1]
rf_scores2  = rf2.predict_proba(X_test2)[:, 1]

results2 = pd.DataFrame({
    'method': ['Week-4 rule baseline', 'Logistic Regression (+query features)', 'Random Forest (+query features)'],
    'precision@20': [precision_at_k(baseline_scores2.values, y_test2, 20),
                      precision_at_k(log_scores2, y_test2, 20),
                      precision_at_k(rf_scores2, y_test2, 20)],
    'precision@50': [precision_at_k(baseline_scores2.values, y_test2, 50),
                      precision_at_k(log_scores2, y_test2, 50),
                      precision_at_k(rf_scores2, y_test2, 50)],
})
print(f"base rate: {y_test2.mean():.3f}")
print(results2)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

base rate: 0.415
                                  method  precision@20  precision@50
0                   Week-4 rule baseline          0.35          0.32
1  Logistic Regression (+query features)          0.85          0.86
2        Random Forest (+query features)          0.95          0.96


The jump to 0.95-0.96 precision was a red flag, not a win. Checking the
query table's schema showed it has separate `_last30` and `_prev30` column
families, and the columns I first used (impressions_90d-based) span the
full 90-day window, which overlaps past the prediction point and into the
outcome window. That's leakage: the model was seeing query performance
from after the moment it's supposed to be predicting at. I rebuilt the
query features using only the `_prev30` columns, which the skill notes
flag as the safe choice.

In [9]:
schema_q = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_query_90d']} LIMIT 1").df()
print(schema_q[['column_name', 'column_type']].to_string())

                      column_name column_type
0                  client_hash_id     VARCHAR
1                 content_hash_id     VARCHAR
2                   query_hash_id     VARCHAR
3                query_char_count      BIGINT
4               query_token_count      BIGINT
5                    window_start        DATE
6                      window_end        DATE
7                 impressions_90d      BIGINT
8                      clicks_90d      BIGINT
9              impressions_last30      BIGINT
10                  clicks_last30      BIGINT
11             impressions_prev30      BIGINT
12                  clicks_prev30      BIGINT
13               avg_position_90d      DOUBLE
14            avg_position_last30      DOUBLE
15            avg_position_prev30      DOUBLE
16  content_total_impressions_90d      BIGINT
17    content_visible_query_count      BIGINT
18               rare_query_count      BIGINT
19         rare_impressions_share      DOUBLE
20   anonymized_impressions_share 

In [10]:
qsignals_safe = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           ANY_VALUE(rare_impressions_share)       AS rare_share,
           SUM(impressions_prev30)                 AS imp_prev30_query,
           MAX(impressions_prev30)                 AS top_query_impressions_prev30
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()
qsignals_safe['top_query_share_prev30'] = (
    qsignals_safe['top_query_impressions_prev30'] / qsignals_safe['imp_prev30_query']
)

data3 = data.merge(qsignals_safe, on='content_hash_id', how='left').dropna(
    subset=['visible_queries', 'rare_share', 'top_query_share_prev30'])

train3 = data3[data3['client_hash_id'].isin(train['client_hash_id'])]
test3  = data3[data3['client_hash_id'].isin(test['client_hash_id'])]

feature_cols3 = ['imp_prev30', 'pos_prev30', 'visible_queries', 'rare_share', 'top_query_share_prev30']
X_train3, y_train3 = train3[feature_cols3], train3['is_declining']
X_test3,  y_test3  = test3[feature_cols3],  test3['is_declining']

log_reg3 = LogisticRegression(max_iter=1000).fit(X_train3, y_train3)
rf3 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X_train3, y_train3)

baseline_scores3 = test3.apply(lambda r: rule_score(r['imp_prev30'], r['pos_prev30']), axis=1)
log_scores3 = log_reg3.predict_proba(X_test3)[:, 1]
rf_scores3  = rf3.predict_proba(X_test3)[:, 1]

results3 = pd.DataFrame({
    'method': ['Week-4 rule baseline', 'Logistic Regression (safe query features)', 'Random Forest (safe query features)'],
    'precision@20': [precision_at_k(baseline_scores3.values, y_test3, 20),
                      precision_at_k(log_scores3, y_test3, 20),
                      precision_at_k(rf_scores3, y_test3, 20)],
    'precision@50': [precision_at_k(baseline_scores3.values, y_test3, 50),
                      precision_at_k(log_scores3, y_test3, 50),
                      precision_at_k(rf_scores3, y_test3, 50)],
})
print(f"base rate: {y_test3.mean():.3f}")
print(results3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

base rate: 0.414
                                      method  precision@20  precision@50
0                       Week-4 rule baseline          0.35          0.28
1  Logistic Regression (safe query features)          0.95          0.92
2        Random Forest (safe query features)          0.95          0.94


In [11]:
window_check = con.sql(f"""
    SELECT DISTINCT window_start, window_end
    FROM {TABLES['fact_query_90d']}
    LIMIT 5
""").df()
print(window_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  window_start window_end
0   2026-04-02 2026-06-30


The 0.92-0.95 scores were leakage, confirmed this time: fact_content_query_90d
is a single fixed window (window_start=2026-04-02, window_end=2026-06-30) for
the entire dataset, not a rolling per-month window. Even the "_prev30" columns
inside it describe April, the exact month I'm using as my outcome window, not
March. There's no version of this table aligned to March, so I can't safely
use it as a feature here at all. I'm dropping the query-mix features entirely
for this model and keeping the original two-feature result (imp_prev30,
pos_prev30) as the honest, final comparison against the Week-4 baseline.

### Attempt 3 — final model: momentum features built safely from fact_daily only
*(query table dropped entirely after the leakage check above; these features
use only pre-April data from the same table used in Week 4, no new source.)*

In [12]:
data_rich = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']} WHERE month='2026-03'),
    prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_clicks ELSE 0 END) AS clk_prev30,
               AVG(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_avg_position END) AS pos_prev30,
               -- second-to-last 30 days (days 31-60 before the cutoff)
               SUM(CASE WHEN report_date <= b.end_d - INTERVAL 30 DAY
                        AND report_date > b.end_d - INTERVAL 60 DAY
                        THEN gsc_impressions ELSE 0 END) AS imp_prev60_30,
               STDDEV(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_avg_position END) AS pos_volatility
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month IN ('2026-02', '2026-03')
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100 AND imp_prev60_30 > 0
    ),
    outcome AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS imp_next30
        FROM {TABLES['fact_daily']} WHERE month = '2026-04'
        GROUP BY 1
    )
    SELECT p.*, o.imp_next30
    FROM prior p JOIN outcome o USING (content_hash_id)
""").df()

data_rich['is_declining'] = (data_rich['imp_next30'] < 0.8 * data_rich['imp_prev30']).astype(int)
data_rich['ctr_prev30'] = data_rich['clk_prev30'] / data_rich['imp_prev30']
data_rich['momentum_ratio'] = data_rich['imp_prev30'] / data_rich['imp_prev60_30']

print(f"{len(data_rich):,} rows, {data_rich['client_hash_id'].nunique()} clients")
print(data_rich['is_declining'].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

86,394 rows, 37 clients
is_declining
1    45827
0    40567
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

gss = GroupShuffleSplit(test_size=0.25, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(data_rich, groups=data_rich['client_hash_id']))
train_r, test_r = data_rich.iloc[train_idx], data_rich.iloc[test_idx]
print(f"Train: {len(train_r):,} rows, {train_r['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test_r):,} rows, {test_r['client_hash_id'].nunique()} clients")

feature_cols_rich = ['imp_prev30', 'pos_prev30', 'ctr_prev30', 'momentum_ratio', 'pos_volatility']
X_train_r = train_r[feature_cols_rich].fillna(0)
y_train_r = train_r['is_declining']
X_test_r  = test_r[feature_cols_rich].fillna(0)
y_test_r  = test_r['is_declining']

def precision_at_k(scores, y_true, k):
    order = scores.argsort()[::-1][:k]
    return y_true.iloc[order].mean()

def rule_score(imp, pos):
    volume_score = 1 - abs(imp - 500) / 5000
    position_score = max(0, (50 - pos) / 50) if pos <= 50 else 0
    return 0.5 * volume_score + 0.5 * position_score

baseline_scores_r = test_r.apply(lambda r: rule_score(r['imp_prev30'], r['pos_prev30']), axis=1)

log_reg_r = LogisticRegression(max_iter=1000).fit(X_train_r, y_train_r)
log_scores_r = log_reg_r.predict_proba(X_test_r)[:, 1]

rf_r = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X_train_r, y_train_r)
rf_scores_r = rf_r.predict_proba(X_test_r)[:, 1]

results_final = pd.DataFrame({
    'method': ['Week-4 rule baseline', 'Logistic Regression (momentum features)', 'Random Forest (momentum features)'],
    'precision@20': [precision_at_k(baseline_scores_r.values, y_test_r, 20),
                      precision_at_k(log_scores_r, y_test_r, 20),
                      precision_at_k(rf_scores_r, y_test_r, 20)],
    'precision@50': [precision_at_k(baseline_scores_r.values, y_test_r, 50),
                      precision_at_k(log_scores_r, y_test_r, 50),
                      precision_at_k(rf_scores_r, y_test_r, 50)],
})
print(f"base rate: {y_test_r.mean():.3f}")
print(results_final)

Train: 59,945 rows, 27 clients
Test:  26,449 rows, 10 clients
base rate: 0.706
                                    method  precision@20  precision@50
0                     Week-4 rule baseline          0.70          0.72
1  Logistic Regression (momentum features)          0.35          0.52
2        Random Forest (momentum features)          0.85          0.80


Adding momentum features (ctr_prev30, momentum_ratio, pos_volatility), all
computed strictly from pre-April data, Random Forest beat the Week-4 rule
baseline honestly: precision@20 = 0.85 vs 0.70, precision@50 = 0.80 vs 0.72.
Logistic Regression underperformed the rule (0.35 at @20), likely because
the real pattern isn't linear, Random Forest can combine conditions (e.g.
low momentum AND high position volatility together) that a linear model
can't express. One caveat: this test split has only 10 clients, so the
base rate (0.706) differs notably from earlier splits (0.471) purely due
to which clients landed in the test group, a real limitation of a
small-panel group split worth flagging rather than hiding.

### "##############################################################################"

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [14]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf_r, X_test_r, y_test_r, n_repeats=10, random_state=42, scoring='precision')
importance_df = pd.DataFrame({
    'feature': feature_cols_rich,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)
print(importance_df)

# A few concrete wrong cases
test_r_copy = test_r.copy()
test_r_copy['predicted_prob'] = rf_scores_r
test_r_copy['predicted_label'] = (rf_scores_r > 0.5).astype(int)
wrong_cases = test_r_copy[test_r_copy['predicted_label'] != test_r_copy['is_declining']]
print(f"\n{len(wrong_cases)} wrong predictions out of {len(test_r_copy)}")
wrong_cases[['content_hash_id', 'imp_prev30', 'pos_prev30', 'momentum_ratio',
             'is_declining', 'predicted_prob']].head(3)

          feature  importance
2      ctr_prev30    0.050160
4  pos_volatility    0.040987
1      pos_prev30    0.021719
0      imp_prev30    0.017314
3  momentum_ratio    0.009159

12401 wrong predictions out of 26449


,content_hash_id,imp_prev30,pos_prev30,momentum_ratio,is_declining,predicted_prob
1072,content_a9ad49ad3d5cc8c9,401.0,35.442670,0.780156,1,0.265
1075,content_88edcb2c7169671d,163.0,41.921369,0.492447,1,0.270
1077,content_d207e75c6052588b,19182.0,4.969266,1.099570,1,0.190


Permutation importance showed ctr_prev30 and pos_volatility mattered most,
more than momentum_ratio, the feature I designed the experiment around.
Stability of performance (low volatility, healthy CTR) seems to carry more
signal than the direction of recent change alone.

Looking at wrong predictions, all three examples shown are false negatives:
pages the model was confident wouldn't decline, but did. The clearest case
had strong signals across the board, high volume (19,182 impressions),
excellent position (4.97), and slightly positive momentum, yet still
declined. This suggests some declines are driven by factors outside these
five features entirely (a competitor's new content, a seasonal shift, an
algorithm update) that no amount of tuning on this feature set alone can
capture. That's a real limit of the model, not just an error to fix by
adding more depth.